# libraries


In [ ]:
using OffsetArrays, Plots, LinearAlgebra, Statistics, Revise, ProgressMeter,JLD2, LaTeXStrings
include("oggetti.jl")
include("gillespie_temp.jl")
path_fig = "images/one_bottleneck_current/"
path_data = "data/currents_vs_q/"


# current as a function of q

GOAL: write a function that:
- takes qmin and qmax with nq points
- lets you choose among the 3 methods (MF, PA, stat)


In [ ]:
function current_q(qmin, qmax, nq;
    methods=["MF", "PA", "stat", "ISA", "GILL", "TRI"], L=200, α=0.2, β=0.3, defect1_length=1, Nt=1e7, Δt=1, tol=1e-11,
    from::Symbol=:center,
    distance_from_boundary::Int64=1, n_defects::Int64=1, q_defect2::Real=1.0, defect2_length::Int64=0, distance_between_defects::Int64=0,
    n_simu=1e3, total_time=800)

    q_values = range(qmin, qmax, length=nq)
    currents = Dict{String,Vector{Float64}}()

    for method in methods
        currents[method] = Float64[]
    end

    begin @inbounds @showprogress for q_defect in q_values

            # define the lattice 
            q = lattice(L, q_defect, defect1_length, from; dbc=distance_from_boundary, n_defects=n_defects, q_defect2=q_defect2, l2=defect2_length, dl=distance_between_defects)
            tasep = Tasep(L, α, β, q)  # example tasep with variable q
            cond_init = (0.5, 0.0)  # example initial conditions

            if "MF" in methods
                ρ, J = EuleroMF(tasep, cond_init; Nt=Nt, Δt=Δt, tol=tol)
                push!(currents["MF"], mean(J))
            end

            if "PA" in methods
                ρ, J = EuleroPA(tasep, cond_init; Nt=Nt, Δt=Δt, tol=tol)
                push!(currents["PA"], mean(J))
            end

            if "stat" in methods
                ρ, J = stationaryMF(tasep, cond_init; verbose=false)
                push!(currents["stat"], mean(J))
            end

            if "ISA" in methods
                J = JisaN(q_defect, defect1_length)
                push!(currents["ISA"], J)
            end
            if "GILL" in methods
                _, J =  Gillespie_temp(tasep, cond_init; n_measures=1e7, termalization_time=10_000, processi_nn=nn(tasep.L))
                push!(currents["GILL"], mean(J))
            end
            if "TRI" in methods
                ρ, J, _, _ = EuleroTRI(tasep, (0.5, 0.25, 0.25, 0.25); Nt=Nt, Δt=Δt, tol=tol)
                push!(currents["TRI"], mean(J))
            end

        end
    end
    return q_values, currents

end

# comparison with the theoretical current computed via PA


In [ ]:
qmin, qmax, nq = 0.1, 1.0, 25

L = 200
α = 0.25
β = 0.25
# q_values, currents = current_q(qmin, qmax, nq; L=L, α=α, β=β);


In [ ]:

ρtheoPA(q) = (2 - 3 * q + sqrt((3q - 2)^2 + 8 * q)) / 4 
JtheoPA(q) = ρtheoPA(q) * (1 - ρtheoPA(q)) # formula for the PA density, computed theoretically


In [ ]:
#@save PATH(L, α, β, n_defects, from) q_values currents
pathh = "data/currents_vs_q/L_200_upMC_lunghezzadifetto_1_from_center.jld2"
@load pathh q_values currents

# lattice with a central defect


In [ ]:
qmin, qmax, nq = 0.1, 1.0, 25
L=200
α=0.6
β=0.7
Nt=1e6
Δt=0.1
tol=1e-11
from=:center
defect1_length=1
n_defects=1
distance_from_boundary=1
q_defect2=1.0
defect2_length=0
distance_between_defects=0
n_simu=10_000

#=
q_values, currents = current_q(qmin, qmax, nq;
    L=L, α=α, β=β,
    methods=["MF", "PA", "ISA", "GILL", "TRI"],
    Nt=Nt, Δt=Δt, tol=tol,
    from=from,
    defect1_length=defect1_length, n_defects=n_defects,
    distance_from_boundary=distance_from_boundary, q_defect2=q_defect2,
    defect2_length=defect2_length, distance_between_defects=distance_between_defects, n_simu=n_simu)
=#

In [ ]:
using JLD2
#@save PATH(L, α, β, n_defects, from) q_values currents
pathh = "data/currents_vs_q/L_200_upMC_lunghezzadifetto_1_from_center.jld2"
@load pathh q_values currents

p = plot(xlabel="q",
    ylabel="J",
    lw=2,
    legend=false,
    legendfontsize=15,
    guidefontsize=20,      # <-- axis label size (xlabel, ylabel)
    tickfontsize=16,
    left_margin=2Plots.mm,     # margin so the large ylabel isn't cut off
    bottom_margin=3Plots.mm)
lw = 2
plot!(p, q_values, currents["MF"], label="MF approximation", color=:dodgerblue, lw=lw)
plot!(p, q_values, currents["ISA"], label="ISA method", color=:black, marker=:circle, markersize=3, markerstrokewidth=0, lw=lw)
plot!(p, q_values, currents["PA"], label="Pair approximation", color=:darkorange, lw=lw)
plot!(p, q_values, currents["TRI"], label="Triplet approximation", color=:green3, lw=lw)
plot!(p, q_values, currents["GILL"], label="Kinetic Monte Carlo", color=:crimson, lw=lw, ls=:dot)

In [ ]:
name_imm = "corrente_vs_q_l_1_all_method.png"
path_pp = "images/one_bottleneck_current_pp/"
savefig(p, path_pp * name_imm)

In [ ]:
# Find the indices of q in the interval [0.4, 0.6]
zoom_range = findall((q_values .>= 0.4) .& (q_values .<= 0.6))

# Use the SAME q values as the x-axis (not a separate range)
zoom_qs = q_values[zoom_range]

# Standalone plot of just the inset (zoom)
p_inset = plot(xlabel="q", ylabel="J",
    title="",
    legend=false,
    titlefontsize=10,
    ylims=(0.2, 0.25),
    framestyle=:box,
    guidefontsize=20,      # <-- axis label size (xlabel, ylabel)
    tickfontsize=16,
    left_margin=2Plots.mm,     # margin so the large ylabel isn't cut off
    bottom_margin=3Plots.mm)
lw = 3

plot!(p_inset, zoom_qs, currents["MF"][zoom_range], lw=lw)
plot!(p_inset, zoom_qs, currents["GILL"][zoom_range], color=:crimson, lw=lw, ls=:dot)
plot!(p_inset, zoom_qs, currents["ISA"][zoom_range], color=:black, marker=:circle, markersize=5, markerstrokewidth=0, lw=lw)
plot!(p_inset, zoom_qs, currents["PA"][zoom_range], color=:darkorange, lw=lw)
plot!(p_inset, zoom_qs, currents["TRI"][zoom_range], color=:green3, lw=lw)




In [ ]:
name_imm = "inset_corrente_vs_q_l_1_all_method.png"
path_pp = "images/one_bottleneck_current_pp/"
savefig(p_inset, path_pp * name_imm)

In [ ]:
delta_isa= currents["GILL"] - currents["ISA"]
sum(delta_isa)
delta_tri= currents["GILL"] - currents["TRI"]
sum(delta_tri)
delta_isa_tri= currents["ISA"] - currents["TRI"]
sum(delta_isa_tri)
[delta_isa[1] delta_tri[1] delta_isa_tri[1]]

In [ ]:
path = "data/currents_vs_q/"
name_data = "L_200_upMC_lunghezzadifetto_1_from_center.jld2"
@load path*name_data q_values currents

deltaJ = abs.(currents["PA"] - currents["ISA"])
scatter(q_values, log10.(deltaJ), xlabel="q", ylabel=L"\log_{10}(\Delta J)", markerwidhtstroke=0, markersize=3, markercolor=:black, label="", size=(600, 350))


In [ ]:
name_imm = "differenza_PA_ISA.png"
savefig(p, path_fig * name_imm)
maximum(abs.(JtheoPA.(q_values) - currents["ISA"]))

# lattice with 2 central defects


In [ ]:
qmin, qmax, nq = 0.1, 1.0, 25
L=200
α=0.8
β=0.7
Nt=1e6
Δt=0.1
tol=1e-11
from=:center
defect1_length=2
n_defects=1
distance_from_boundary=1
q_defect2=1.0
defect2_length=0
distance_between_defects=0
n_simu=10_000
total_time=800
#=
q_values, currents = current_q(qmin, qmax, nq;
    L=L, α=α, β=β,
    Nt=Nt, Δt=Δt, tol=tol,
    from=from,
    defect1_length=defect1_length, n_defects=n_defects,
    distance_from_boundary=distance_from_boundary, q_defect2=q_defect2,
    defect2_length=defect2_length, distance_between_defects=distance_between_defects, n_simu=n_simu, total_time=total_time )

=#


In [ ]:
path_data = "data/currents_vs_q/"
name_data = "L_200_upMC_lunghezzadifetto_2_from_center.jld2" 
#@save path_data*name_data q_values currents
@load path_data*name_data q_values currents


p = plot(xlabel="q",
    ylabel="J",
    lw=2,
    legend=:bottomright)
plot!(p, q_values, currents["ISA"], label="ISA method",color=:black, marker=:circle , markersize=2, markerstrokewidth= 0)
plot!(p, q_values, currents["MF"], label="MF approximation",color=:dodgerblue)
plot!(p, q_values, currents["PA"], label="Pair approximation", color=:darkorange)
plot!(p, q_values, currents["TRI"], label="Triplet approximation", color=:green3)
plot!(p, q_values, currents["GILL"], label="Numerical solution", color=:crimson, lw=1, ls=:dot)


# zoom
 zoom_min, zoom_max = 0.5, 0.6
    zoom_range = findall(q-> zoom_min <= q <= zoom_max, q_values)
    zoom_q = q_values[zoom_range]

    x, y = 0.65, 0.25
    width, height = 0.3, 0.4

    # Only create the inset, nothing else
    plot!(p, inset=(1, bbox(x, y, width, height)))

    plot!(p[2], zoom_q, currents["MF"][zoom_range], label="",
        legend=false,
        titlefontsize=8,
        tickfontsize=6,
        ylims=(0.2, 0.23),
        xlims=(zoom_min, zoom_max),
        framestyle=:box,
        marker=:circle, markersize=3, markerstrokewidth=0)
    plot!(p[2], zoom_q, currents["PA"][zoom_range], label="", color=:darkorange,
        marker=:circle, markersize=3, markerstrokewidth=0)
    plot!(p[2], zoom_q, currents["TRI"][zoom_range], label="", color=:green3,
        marker=:circle, markersize=3, markerstrokewidth=0)
    plot!(p[2], zoom_q, currents["GILL"][zoom_range], label="", color=:crimson, lw=1, ls=:dot,
        marker=:circle, markersize=2, markerstrokewidth=0)
    plot!(p[2],zoom_q, currents["ISA"][zoom_range], label="",color=:black, marker=:circle , markersize=2, markerstrokewidth= 0)


In [ ]:
name_imm = "corrente_vs_q_l_2_all_method.png"
savefig(p, path_fig * name_imm)

# 3 central defects


In [ ]:
qmin, qmax, nq = 0.1, 1.0, 25
L=200
α=0.8
β=0.7
Nt=1e6
Δt=0.1
tol=1e-11
from=:center
defect1_length=3
n_defects=1
distance_from_boundary=1
q_defect2=1.0
defect2_length=0
distance_between_defects=0
n_simu=10_000
total_time=800

#=
q_values, currents = current_q(qmin, qmax, nq;
    L=L, α=α, β=β,
    Nt=Nt, Δt=Δt, tol=tol,
    from=from,
    defect1_length=defect1_length, n_defects=n_defects,
    distance_from_boundary=distance_from_boundary, q_defect2=q_defect2,
    defect2_length=defect2_length, distance_between_defects=distance_between_defects, n_simu=n_simu, total_time=total_time )
=#

In [ ]:
path_data = "data/currents_vs_q/"
name_data_l3 = "L_200_upMC_lunghezzadifetto_3_from_center.jld2"

@save  path_data*name_data_l3 q_values currents
#@load path_data*name_data_l3 q_values currents


p = plot(xlabel="q",
    ylabel="J",
    lw=2,
    legend=:bottomright)
plot!(p, q_values, currents["ISA"], label="",color=:black, marker=:circle , markersize=2, markerstrokewidth= 0)
plot!(p, q_values, currents["MF"], label="",color=:dodgerblue)
plot!(p, q_values, currents["PA"], label="", color=:darkorange)
plot!(p, q_values, currents["TRI"], label="", color=:green3)
plot!(p, q_values, currents["GILL"], label="", color=:crimson, lw=1, ls=:dot)
# zoom
 zoom_min, zoom_max = 0.72, 0.87
    zoom_range = findall(q-> zoom_min <= q <= zoom_max, q_values)
    zoom_q = q_values[zoom_range]

    x, y = 0.67, 0.39
    width, height = 0.3, 0.4

    # Only create the inset, nothing else
    plot!(p, inset=(1, bbox(x, y, width, height)))

    plot!(p[2], zoom_q, currents["MF"][zoom_range], label="",
        legend=false,
        titlefontsize=8,
        tickfontsize=6,
        ylims=(0.22, 0.25),
        xlims=(zoom_min, zoom_max),
        framestyle=:box,
        marker=:circle, markersize=3, markerstrokewidth=0)
    plot!(p[2], zoom_q, currents["PA"][zoom_range], label="", color=:darkorange,
        marker=:circle, markersize=3, markerstrokewidth=0)
    plot!(p[2], zoom_q, currents["TRI"][zoom_range], label="", color=:green3,
        marker=:circle, markersize=3, markerstrokewidth=0)
    plot!(p[2], zoom_q, currents["GILL"][zoom_range], label="", color=:crimson, lw=1, ls=:dot,
        marker=:circle, markersize=2, markerstrokewidth=0)
    plot!(p[2],zoom_q, currents["ISA"][zoom_range], label="",color=:black, marker=:circle , markersize=2, markerstrokewidth= 0)


In [ ]:
name_imm = "corrente_vs_q_l_3_all_method.png"
savefig(p, path_fig * name_imm)

In [ ]:
path = "data/currents_vs_q/"
name_data = "L_200_upMC_lunghezzadifetto_3_from_center.jld2"
@load path*name_data q_values currents

deltaJ = abs.(currents["TRI"] - currents["ISA"])
scatter(q_values, log10.(deltaJ), xlabel="q", ylabel=L"\log_{10}(\Delta J)", markerwidhtstroke=0, markersize=3, markercolor=:black, label="", size=(600, 350))

In [ ]:
name_imm = "differenza_TRI_ISA_l3.png"
savefig(p, path_fig * name_imm)

# 4 central defects


In [ ]:
qmin, qmax, nq = 0.1, 1.0, 25
L=200
α=0.8
β=0.7
Nt=1e6
Δt=0.1
tol=1e-11
from=:center
defect1_length=4
n_defects=1
distance_from_boundary=1
q_defect2=1.0
defect2_length=0
distance_between_defects=0
n_simu=10_000
total_time=800


# q_values, currents = current_q(qmin, qmax, nq;
#     L=L, α=α, β=β,
#     Nt=Nt, Δt=Δt, tol=tol,
#     from=from,
#     defect1_length=defect1_length, n_defects=n_defects,
#     distance_from_boundary=distance_from_boundary, q_defect2=q_defect2,
#     defect2_length=defect2_length, distance_between_defects=distance_between_defects, n_simu=n_simu, total_time=total_time )



In [ ]:
path_data = "data/currents_vs_q/"
name_data_l4 = "L_200_upMC_lunghezzadifetto_4_from_center.jld2"

#@save  path_data*name_data_l4 q_values currents
@load path_data*name_data_l4 q_values currents


p = plot(xlabel="q",
    ylabel="J",
    lw=2,
    legend=:bottomright)
plot!(p, q_values, currents["ISA"], label="",color=:black, marker=:circle , markersize=2, markerstrokewidth= 0)
plot!(p, q_values, currents["MF"], label="",color=:dodgerblue)
plot!(p, q_values, currents["PA"], label="", color=:darkorange)
plot!(p, q_values, currents["TRI"], label="", color=:green3)
plot!(p, q_values, currents["GILL"], label="", color=:crimson, lw=1, ls=:dot)
# zoom
 zoom_min, zoom_max = 0.77, 0.85
    zoom_range = findall(q-> zoom_min <= q <= zoom_max, q_values)
    zoom_q = q_values[zoom_range]

    x, y = 0.67, 0.35
    width, height = 0.3, 0.4

    # Only create the inset, nothing else
    plot!(p, inset=(1, bbox(x, y, width, height)))

    plot!(p[2], zoom_q, currents["MF"][zoom_range], label="",
        legend=false,
        titlefontsize=8,
        tickfontsize=6,
        ylims=(0.20, 0.25),
        xlims=(zoom_min, zoom_max),
        framestyle=:box,
        marker=:circle, markersize=3, markerstrokewidth=0)
    plot!(p[2], zoom_q, currents["PA"][zoom_range], label="", color=:darkorange,
        marker=:circle, markersize=3, markerstrokewidth=0)
    plot!(p[2], zoom_q, currents["TRI"][zoom_range], label="", color=:green3,
        marker=:circle, markersize=3, markerstrokewidth=0)
    plot!(p[2], zoom_q, currents["GILL"][zoom_range], label="", color=:crimson, lw=1, ls=:dot,
        marker=:circle, markersize=2, markerstrokewidth=0)
    plot!(p[2],zoom_q, currents["ISA"][zoom_range], label="",color=:black, marker=:circle , markersize=2, markerstrokewidth= 0)


In [ ]:
name_imm = "corrente_vs_q_l_4_all_method.png"
savefig(p, path_fig * name_imm)

In [ ]:
pathh = "data/currents_vs_q/"
name_data = "L_200_upMC_lunghezzadifetto_4_from_center.jld2"
@load pathh*name_data q_values currents

deltaJ = abs.(currents["TRI"] - currents["ISA"])
scatter(q_values, log10.(deltaJ), xlabel="q", ylabel=L"\log_{10}(J_{TA}-J_{ISA})",
    markerwidhtstroke=0, markersize=3, markercolor=:black,
    label="", size=(600, 350),
    legendfontsize=13,
    guidefontsize=18,          # label assi
    tickfontsize=16,           # <-- AGGIUNGI: numeri sugli assi (spesso dimenticato)
    left_margin=5Plots.mm,     # margin so the large ylabel isn't cut off
    bottom_margin=5Plots.mm
)

In [ ]:
name_imm = "differenza_TRI_ISA_l4.png"
savefig(p, path_fig * name_imm)

# 5 central defects


In [ ]:
qmin, qmax, nq = 0.1, 1.0, 25
L=200
α=0.8
β=0.7
Nt=1e6
Δt=0.1
tol=1e-11
from=:center
defect1_length=5
n_defects=1
distance_from_boundary=1
q_defect2=1.0
defect2_length=0
distance_between_defects=0
n_simu=10_000
total_time=800

#=
q_values, currents = current_q(qmin, qmax, nq;
    L=L, α=α, β=β,
    Nt=Nt, Δt=Δt, tol=tol,
    from=from,
    defect1_length=defect1_length, n_defects=n_defects,
    distance_from_boundary=distance_from_boundary, q_defect2=q_defect2,
    defect2_length=defect2_length, distance_between_defects=distance_between_defects, n_simu=n_simu, total_time=total_time )



In [ ]:
path_data = "data/currents_vs_q/"
name_data_l5 = "L_200_upMC_lunghezzadifetto_5_from_center.jld2"

#@save  path_data*name_data_l5 q_values currents
@load path_data*name_data_l5 q_values currents


p = plot(xlabel="q",
    ylabel="J",
    lw=2,
    legend=:bottomright)
plot!(p, q_values, currents["ISA"], label="",color=:black, marker=:circle , markersize=2, markerstrokewidth= 0)
plot!(p, q_values, currents["MF"], label="",color=:dodgerblue)
plot!(p, q_values, currents["PA"], label="", color=:darkorange)
plot!(p, q_values, currents["TRI"], label="", color=:green3)
plot!(p, q_values, currents["GILL"], label="", color=:crimson, lw=1, ls=:dot)
# zoom
 zoom_min, zoom_max = 0.77, 0.85
    zoom_range = findall(q-> zoom_min <= q <= zoom_max, q_values)
    zoom_q = q_values[zoom_range]

    x, y = 0.67, 0.35
    width, height = 0.3, 0.4

    # Only create the inset, nothing else
    plot!(p, inset=(1, bbox(x, y, width, height)))

    plot!(p[2], zoom_q, currents["MF"][zoom_range], label="",
        legend=false,
        titlefontsize=8,
        tickfontsize=6,
        ylims=(0.20, 0.25),
        xlims=(zoom_min, zoom_max),
        framestyle=:box,
        marker=:circle, markersize=3, markerstrokewidth=0)
    plot!(p[2], zoom_q, currents["PA"][zoom_range], label="", color=:darkorange,
        marker=:circle, markersize=3, markerstrokewidth=0)
    plot!(p[2], zoom_q, currents["TRI"][zoom_range], label="", color=:green3,
        marker=:circle, markersize=3, markerstrokewidth=0)
    plot!(p[2], zoom_q, currents["GILL"][zoom_range], label="", color=:crimson, lw=1, ls=:dot,
        marker=:circle, markersize=2, markerstrokewidth=0)
    plot!(p[2],zoom_q, currents["ISA"][zoom_range], label="",color=:black, marker=:circle , markersize=2, markerstrokewidth= 0)


In [ ]:
name_imm = "corrente_vs_q_l_5_all_method.png"
savefig(p, path_fig * name_imm)

In [ ]:
path = "data/currents_vs_q/"
name_data = "L_200_upMC_lunghezzadifetto_5_from_center.jld2"
@load path*name_data q_values currents

deltaJ = abs.(currents["TRI"] - currents["ISA"])
scatter(q_values, log10.(deltaJ), xlabel="q", ylabel=L"\log_{10}(\Delta J)", markerwidhtstroke=0, markersize=3, markercolor=:black, label="", size=(600, 350))


In [ ]:
name_imm = "differenza_TRI_ISA_l5.png"
savefig(p, path_fig * name_imm)

# 6 central defects


In [ ]:
qmin, qmax, nq = 0.1, 1.0, 25
L=200
α=0.8
β=0.7
Nt=1e6
Δt=0.1
tol=1e-11
from=:center
defect1_length=6
n_defects=1
distance_from_boundary=1
q_defect2=1.0
defect2_length=0
distance_between_defects=0
n_simu=10_000
total_time=800

#=
q_values, currents = current_q(qmin, qmax, nq;
    L=L, α=α, β=β,
    Nt=Nt, Δt=Δt, tol=tol,
    from=from,
    defect1_length=defect1_length, n_defects=n_defects,
    distance_from_boundary=distance_from_boundary, q_defect2=q_defect2,
    defect2_length=defect2_length, distance_between_defects=distance_between_defects, n_simu=n_simu, total_time=total_time )
   


In [ ]:
path_data = "data/currents_vs_q/"
name_data_l6 = "L_200_upMC_lunghezzadifetto_6_from_center.jld2"

@save  path_data*name_data_l6 q_values currents
#@load path_data*name_data_l6 q_values currents


p = plot(xlabel="q",
    ylabel="J",
    lw=2,
    legend=:bottomright)
plot!(p, q_values, currents["ISA"], label="",color=:black, marker=:circle , markersize=2, markerstrokewidth= 0)
plot!(p, q_values, currents["MF"], label="",color=:dodgerblue)
plot!(p, q_values, currents["PA"], label="", color=:darkorange)
plot!(p, q_values, currents["TRI"], label="", color=:green3)
plot!(p, q_values, currents["GILL"], label="", color=:crimson, lw=1, ls=:dot)
# zoom
 zoom_min, zoom_max = 0.7, 0.9
    zoom_range = findall(q-> zoom_min <= q <= zoom_max, q_values)
    zoom_q = q_values[zoom_range]

    x, y = 0.65, 0.45
    width, height = 0.3, 0.4

    # Only create the inset, nothing else
    plot!(p, inset=(1, bbox(x, y, width, height)))

    plot!(p[2], zoom_q, currents["MF"][zoom_range], label="",
        legend=false,
        titlefontsize=8,
        tickfontsize=6,
        ylims=(0.22, 0.25),
        xlims=(zoom_min, zoom_max),
        framestyle=:box,
        marker=:circle, markersize=3, markerstrokewidth=0)
    plot!(p[2], zoom_q, currents["PA"][zoom_range], label="", color=:darkorange,
        marker=:circle, markersize=3, markerstrokewidth=0)
    plot!(p[2], zoom_q, currents["TRI"][zoom_range], label="", color=:green3,
        marker=:circle, markersize=3, markerstrokewidth=0)
    plot!(p[2], zoom_q, currents["GILL"][zoom_range], label="", color=:crimson, lw=1, ls=:dot,
        marker=:circle, markersize=2, markerstrokewidth=0)
    plot!(p[2],zoom_q, currents["ISA"][zoom_range], label="",color=:black, marker=:circle , markersize=2, markerstrokewidth= 0)




In [ ]:
name_imm = "corrente_vs_q_l_6_all_method.png"
savefig(p, path_fig * name_imm)

In [ ]:
path = "data/currents_vs_q/"
name_data = "L_200_upMC_lunghezzadifetto_6_from_center.jld2"
@load path*name_data q_values currents

deltaJ = abs.(currents["TRI"] - currents["ISA"])
scatter(q_values, log10.(deltaJ), xlabel="q", ylabel=L"\log_{10}(\Delta J)", markerwidhtstroke=0, markersize=3, markercolor=:black, label="", size=(600, 350))

In [ ]:
name_imm = "differenza_PA_ISA_l6.png"
savefig(p, path_fig * name_imm)

# 7 central defects


In [ ]:
qmin, qmax, nq = 0.1, 1.0, 25
L=200
α=0.8
β=0.7
Nt=1e6
Δt=0.1
tol=1e-11
from=:center
defect1_length=7
n_defects=1
distance_from_boundary=1
q_defect2=1.0
defect2_length=0
distance_between_defects=0
n_simu=10_000
total_time=800
#=
q_values, currents = current_q(qmin, qmax, nq;
    L=L, α=α, β=β,
    Nt=Nt, Δt=Δt, tol=tol,
    from=from,
    defect1_length=defect1_length, n_defects=n_defects,
    distance_from_boundary=distance_from_boundary, q_defect2=q_defect2,
    defect2_length=defect2_length, distance_between_defects=distance_between_defects, n_simu=n_simu, total_time=total_time )

=#


In [ ]:
path_data = "data/currents_vs_q/"
name_data_l7 = "L_200_upMC_lunghezzadifetto_7_from_center.jld2"

#@save  path_data*name_data_l7 q_values currents
@load path_data*name_data_l7 q_values currents


p = plot(xlabel="q",
    ylabel="J",
    lw=2,
    legend=:bottomright)
plot!(p, q_values, currents["ISA"], label="",color=:black, marker=:circle , markersize=2, markerstrokewidth= 0)
plot!(p, q_values, currents["MF"], label="",color=:dodgerblue)
plot!(p, q_values, currents["PA"], label="", color=:darkorange)
plot!(p, q_values, currents["TRI"], label="", color=:green3)
plot!(p, q_values, currents["GILL"], label="", color=:crimson, lw=1, ls=:dot)
# zoom
 zoom_min, zoom_max = 0.77, 0.9
    zoom_range = findall(q-> zoom_min <= q <= zoom_max, q_values)
    zoom_q = q_values[zoom_range]

    x, y = 0.65, 0.45
    width, height = 0.3, 0.4

    # Only create the inset, nothing else
    plot!(p, inset=(1, bbox(x, y, width, height)))

    plot!(p[2], zoom_q, currents["MF"][zoom_range], label="",
        legend=false,
        titlefontsize=8,
        tickfontsize=6,
        ylims=(0.22, 0.25),
        xlims=(zoom_min, zoom_max),
        framestyle=:box,
        marker=:circle, markersize=3, markerstrokewidth=0)
    plot!(p[2], zoom_q, currents["PA"][zoom_range], label="", color=:darkorange,
        marker=:circle, markersize=3, markerstrokewidth=0)
    plot!(p[2], zoom_q, currents["TRI"][zoom_range], label="", color=:green3,
        marker=:circle, markersize=3, markerstrokewidth=0)
    plot!(p[2], zoom_q, currents["GILL"][zoom_range], label="", color=:crimson, lw=1, ls=:dot,
        marker=:circle, markersize=2, markerstrokewidth=0)
    plot!(p[2],zoom_q, currents["ISA"][zoom_range], label="",color=:black, marker=:circle , markersize=2, markerstrokewidth= 0)




In [ ]:
name_imm = "corrente_vs_q_l_7_all_method.png"
savefig(p, path_fig * name_imm)

In [ ]:
path = "data/currents_vs_q/"
name_data = "L_200_upMC_lunghezzadifetto_7_from_center.jld2"
@load path*name_data q_values currents

deltaJ = abs.(currents["TRI"] - currents["ISA"])
scatter(q_values, log10.(deltaJ), xlabel="q", ylabel=L"\log_{10}(\Delta J)", markerwidhtstroke=0, markersize=3, markercolor=:black, label="", size=(600, 350))

In [ ]:
name_imm = "differenza_PA_ISA_l7.png"
savefig(p, path_fig * name_imm)

# end
